# Cross-family check: Qwen3-8B and Llama 3.1 8B (preregistered: `docs/preregistration_cross_family.md`)

Does the Gemma account hold in models of two other families? The flow test runs unchanged on each model (behaviour,
probes for both steps along the way, the truth probe at the answer position, logit lens and copying), and a short
summary applies the preregistered criteria.

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. For Llama: open https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct while logged in and accept the licence
   (approval usually takes minutes to a few hours). Without access the notebook runs Qwen3-8B only and says so.
4. Runtime → Change runtime type → **L4** (enough: both models fit whole), A100 or H100. Then **Run all**. About 1–2
   hours per model on an L4, less on an A100.

Results are saved to Drive as they are produced; after a disconnect, just press Run all again (finished models are
skipped). The last cell downloads `cross_family_results.zip`; send that file back.

In [ ]:
# 1. GPU check
import subprocess, torch, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 / A100 / H100"
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
assert GB > 20, f"Only {GB:.0f} GB of GPU memory: pick an L4, A100 or H100"
BS = 16 if GB > 35 else 8   # batch size: memory only
print(f"{GB:.0f} GB: both 8B models fit whole; batch size {BS}")

In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
assert os.path.exists("/content/geomap/scripts/cross_family_summary.py"), "Old bundle: upload the new geomap_colab.zip"
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
os.makedirs(f"{RES}/flow", exist_ok=True)
print("already saved:", sorted(f for f in os.listdir(f"{RES}/flow") if f.startswith(("qwen3-8b", "llama3.1-8b"))))

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' imports)
!pip -q install "transformers==4.57.3" nnsight accelerate
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional); is the gated Llama 3.1 available to this account?
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami, hf_hub_download
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
try:
    hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "config.json", token=os.environ["HF_TOKEN"])
    RUN_LLAMA = True
    print("Llama 3.1 8B: access OK")
except Exception as e:
    RUN_LLAMA = False
    print("Llama 3.1 8B: NO ACCESS (accept the licence on its Hugging Face page, then rerun). Running Qwen3-8B only.")
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[Cross-family] Colab run started (or resumed)")

In [ ]:
# 5. Qwen3-8B (non-thinking mode, as in all earlier Qwen runs)
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/flow_test.py --model qwen3-8b --batch-size {BS} --acts-dir /content/acts 2>&1 | grep -v -i warning

In [ ]:
# 6. Llama 3.1 8B Instruct (only with access to the gated weights)
if RUN_LLAMA:
    !cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/flow_test.py --model llama3.1-8b --batch-size {BS} --acts-dir /content/acts 2>&1 | grep -v -i warning
else:
    print("Skipping Llama 3.1 8B: no access. The verdict will rest on Qwen3-8B alone.")

In [ ]:
# 7. The preregistered criteria
!cd /content/geomap && PYTHONPATH=/content/geomap/src python scripts/cross_family_summary.py

In [ ]:
# 8. One zip of the results + the results log, on Drive and downloaded here
import glob
with zipfile.ZipFile("/content/cross_family_results.zip", "w") as z:
    for f in glob.glob(f"{RES}/flow/qwen3-8b_flow*") + glob.glob(f"{RES}/flow/llama3.1-8b_flow*") + glob.glob(f"{RES}/flow/cross_family.done"):
        if not f.endswith("_acts.pt"):
            z.write(f, "results/flow/" + os.path.basename(f))
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
    print("\n".join(z.namelist()))
shutil.copy("/content/cross_family_results.zip", f"{D}/cross_family_results.zip")
notify("[Cross-family] zip saved: MyDrive/geomap/cross_family_results.zip")
from google.colab import files
files.download("/content/cross_family_results.zip")